# 02 - Preprocess Dataset

Loads the dataset via `datasets.load_dataset()` (the same function used internally by `inference/generate_reports.py`), validates that all referenced images exist and are readable, and reports basic corpus statistics for the reference reports.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/medical-report-benchmark')

from utils.io import load_config
from datasets import load_dataset

config = load_config('configs/default.yaml')
df = load_dataset(config)
print(f'Loaded {len(df)} samples')
df.head()

In [ ]:
# Validate that every image path is readable; drop samples with missing/corrupt images.
from utils.image import load_image
from tqdm import tqdm

valid_mask = []
for path in tqdm(df['image_path'], desc='validating images'):
    img = load_image(path)
    valid_mask.append(img is not None)

df['image_valid'] = valid_mask
print('Valid images:', df['image_valid'].sum(), '/', len(df))
clean_df = df[df['image_valid']].drop(columns=['image_valid']).reset_index(drop=True)

In [ ]:
# Basic report-length statistics -- useful for sanity-checking the dataset
# and for choosing a reasonable max_new_tokens value in configs/default.yaml.
report_lengths = clean_df['ground_truth_report'].str.split().apply(len)
print(report_lengths.describe())

In [ ]:
# Persist the cleaned, validated dataframe for downstream notebooks to reuse directly
# (avoids re-validating images in every subsequent notebook).
clean_df.to_csv('/kaggle/working/medical-report-benchmark/outputs/clean_dataset.csv', index=False)
print('Saved cleaned dataset with', len(clean_df), 'samples')